### Electric power


In [0]:
from pyspark.sql.functions import col

electricity_power = (
    spark.table("smartmeter.gold.utility_usage")
    .filter(
        (col("meter_type") == "electricity") &
        (col("parameter_name") == "Active power")
    )
)

display(electricity_power)

### electricity_power_analysis table

In [0]:
from pyspark.sql.functions import col, to_date, hour

electricity_power_analysis = (
    electricity_power
    .withColumn("reading_date", to_date("timestamp"))
    .withColumn("reading_hour", hour("timestamp"))
    .select(
        "meter_id",
        "area_name",
        "timestamp",
        "reading_date",
        "reading_hour",
        col("value").alias("active_power_kw"),
        "unit",
        "quality",
        "manufacturer",
        "model"
    )
)

display(electricity_power_analysis)

### saving to gold layer

In [0]:
electricity_power_analysis.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("smartmeter.gold.electricity_power_analysis")

### hourly electricity power analysis

In [0]:
from pyspark.sql.functions import (
    col,
    avg,
    max,
    min,
    count,
    date_trunc
)

hourly_electricity = (
    electricity_power_analysis
    .groupBy(
        "meter_id",
        "area_name",
        date_trunc("hour", "timestamp").alias("hour")
    )
    .agg(
        avg("active_power_kw").alias("avg_power_kw"),
        max("active_power_kw").alias("peak_power_kw"),
        min("active_power_kw").alias("min_power_kw"),
        count("*").alias("reading_count")
    )
    .orderBy("hour", "meter_id")
)

display(hourly_electricity)

In [0]:
hourly_electricity.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("smartmeter.gold.hourly_electricity_power")

### Meter-wise Power Analysis

In [0]:
from pyspark.sql.functions import avg, max, min, count

meter_electricity_analysis = (
    electricity_power_analysis
    .groupBy(
        "meter_id",
        "area_name"
    )
    .agg(
        avg("active_power_kw").alias("avg_power_kw"),
        max("active_power_kw").alias("peak_power_kw"),
        min("active_power_kw").alias("min_power_kw"),
        count("*").alias("reading_count")
    )
    .orderBy(col("peak_power_kw").desc())
)

display(meter_electricity_analysis)

In [0]:
meter_electricity_analysis.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("smartmeter.gold.electricity_meter_analysis")

### Area-wise electricity power analysis

In [0]:
from pyspark.sql.functions import avg, max, min, count

area_electricity_analysis = (
    electricity_power_analysis
    .groupBy("area_name")
    .agg(
        avg("active_power_kw").alias("avg_power_kw"),
        max("active_power_kw").alias("peak_power_kw"),
        min("active_power_kw").alias("min_power_kw"),
        count("*").alias("reading_count")
    )
    .orderBy(col("peak_power_kw").desc())
)

display(area_electricity_analysis)

In [0]:
area_electricity_analysis.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("smartmeter.gold.electricity_area_analysis")

### Peak Power Analysis

In [0]:
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

peak_window = Window.partitionBy("meter_id").orderBy(
    col("active_power_kw").desc()
)

peak_power_analysis = (
    electricity_power_analysis
    .withColumn("rank", row_number().over(peak_window))
    .filter(col("rank") == 1)
    .select(
        "meter_id",
        "area_name",
        "timestamp",
        "active_power_kw",
        "quality",
        "manufacturer",
        "model"
    )
    .orderBy(col("active_power_kw").desc())
)

display(peak_power_analysis)

In [0]:
peak_power_analysis.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("smartmeter.gold.electricity_peak_power")